In [ ]:
import pickle
import numpy as np
import pandas as pd
import joblib

# ============================================================
# LOAD MODEL
# ============================================================
model = joblib.load('models/neural_net_model_20251029.pkl')

print("✅ Model loaded successfully!")
print(f"Model type: {type(model).__name__}")

# ============================================================
# FEATURE DEFINITIONS (must match training features!)
# ============================================================
FEATURES = [
    # === CORE ELO FEATURES ===
    'team1_elo_before', 'team2_elo_before', 'elo_diff',
    
    # === HEAD-TO-HEAD ===
    'h2h_winrate',
    'h2h_games_played',
    'h2h_avg_score_diff',
    
    # === RECENT FORM - Mixed time windows ===
    'team1_winrate_3', 'team2_winrate_3',
    'team1_winrate_10', 'team2_winrate_10',
    'team1_winrate_30d', 'team2_winrate_30d',
    
    # === ELO CONTEXT & MOMENTUM ===
    'team1_elo_momentum', 'team2_elo_momentum',
    'team1_opponent_elo_mean', 'team2_opponent_elo_mean',
    
    # === BETTING MARKET SIGNALS ===
    'market_margin',
    
    # === MATCH CONTEXT ===
    'bo_type',
    'is_playoffs',
    'is_international',
    'team1_tournament_games', 'team2_tournament_games',
    
    # === SIDE PREFERENCE ===
    'team1_blue_winrate', 'team1_red_winrate',
    'team2_blue_winrate', 'team2_red_winrate',
    
    # === REST & FATIGUE ===
    'team1_days_rest', 'team2_days_rest',
    'team1_games_30d', 'team2_games_30d',
    
    # === IN-GAME PERFORMANCE ===
    'team1_vision_control', 'team2_vision_control',
    'team1_dragon_control', 'team2_dragon_control',
    'team1_baron_control', 'team2_baron_control'
]

# ============================================================
# PREDICTION FUNCTION
# ============================================================
def predict_match(
    # CORE ELO FEATURES
    team1_elo_before=1500,
    team2_elo_before=1500,
    
    # HEAD-TO-HEAD
    h2h_winrate=0.5,
    h2h_games_played=0,
    h2h_avg_score_diff=0,
    
    # RECENT FORM
    team1_winrate_3=0.5,
    team2_winrate_3=0.5,
    team1_winrate_10=0.5,
    team2_winrate_10=0.5,
    team1_winrate_30d=0.5,
    team2_winrate_30d=0.5,
    
    # ELO CONTEXT & MOMENTUM
    team1_elo_momentum=0,
    team2_elo_momentum=0,
    team1_opponent_elo_mean=1500,
    team2_opponent_elo_mean=1500,
    
    # BETTING MARKET SIGNALS
    market_margin=0.05,
    
    # MATCH CONTEXT
    bo_type=3,
    is_playoffs=0,
    is_international=0,
    team1_tournament_games=10,
    team2_tournament_games=10,
    
    # SIDE PREFERENCE
    team1_blue_winrate=0.5,
    team1_red_winrate=0.5,
    team2_blue_winrate=0.5,
    team2_red_winrate=0.5,
    
    # REST & FATIGUE
    team1_days_rest=3,
    team2_days_rest=3,
    team1_games_30d=15,
    team2_games_30d=15,
    
    # IN-GAME PERFORMANCE
    team1_vision_control=1.0,
    team2_vision_control=1.0,
    team1_dragon_control=0.5,
    team2_dragon_control=0.5,
    team1_baron_control=0.5,
    team2_baron_control=0.5,
    
    # Odds (optional, for value calculation)
    team1_odds=None,
    team2_odds=None
):
    """
    Predict match outcome and calculate betting value
    
    Returns:
        dict with probabilities, prediction, and betting recommendations
    """
    
    # Calculate derived features
    elo_diff = team1_elo_before - team2_elo_before
    
    # Create feature array
    feature_values = [
        # CORE ELO FEATURES
        team1_elo_before, team2_elo_before, elo_diff,
        
        # HEAD-TO-HEAD
        h2h_winrate, h2h_games_played, h2h_avg_score_diff,
        
        # RECENT FORM
        team1_winrate_3, team2_winrate_3,
        team1_winrate_10, team2_winrate_10,
        team1_winrate_30d, team2_winrate_30d,
        
        # ELO CONTEXT & MOMENTUM
        team1_elo_momentum, team2_elo_momentum,
        team1_opponent_elo_mean, team2_opponent_elo_mean,
        
        # BETTING MARKET SIGNALS
        market_margin,
        
        # MATCH CONTEXT
        bo_type, is_playoffs, is_international,
        team1_tournament_games, team2_tournament_games,
        
        # SIDE PREFERENCE
        team1_blue_winrate, team1_red_winrate,
        team2_blue_winrate, team2_red_winrate,
        
        # REST & FATIGUE
        team1_days_rest, team2_days_rest,
        team1_games_30d, team2_games_30d,
        
        # IN-GAME PERFORMANCE
        team1_vision_control, team2_vision_control,
        team1_dragon_control, team2_dragon_control,
        team1_baron_control, team2_baron_control
    ]
    
    # Create DataFrame (model expects this format)
    X = pd.DataFrame([feature_values], columns=FEATURES)
    
    # Predict
    prediction = model.predict(X)[0]
    probabilities = model.predict_proba(X)[0]
    
    team1_prob = probabilities[1]  # Probability team1 wins
    team2_prob = probabilities[0]  # Probability team2 wins
    
    # Result dictionary
    result = {
        'team1_win_probability': team1_prob,
        'team2_win_probability': team2_prob,
        'predicted_winner': 'Team 1' if prediction == 1 else 'Team 2',
        'confidence': max(team1_prob, team2_prob)
    }
    
    # Calculate betting value if odds provided
    if team1_odds and team2_odds:
        team1_expected_value = team1_odds * team1_prob - 1
        team2_expected_value = team2_odds * team2_prob - 1
        
        team1_implied_prob = 1 / team1_odds
        team2_implied_prob = 1 / team2_odds
        
        result['betting_analysis'] = {
            'team1': {
                'odds': team1_odds,
                'model_probability': team1_prob,
                'implied_probability': team1_implied_prob,
                'expected_value': team1_expected_value,
                'edge': team1_prob - team1_implied_prob,
                'recommendation': 'BET' if 0.03 < team1_expected_value < 0.50 and 0.5 < team1_prob < 0.9 else 'SKIP'
            },
            'team2': {
                'odds': team2_odds,
                'model_probability': team2_prob,
                'implied_probability': team2_implied_prob,
                'expected_value': team2_expected_value,
                'edge': team2_prob - team2_implied_prob,
                'recommendation': 'BET' if 0.03 < team2_expected_value < 0.50 and 0.5 < team2_prob < 0.9 else 'SKIP'
            }
        }
    
    return result


# ============================================================
# PRETTY PRINT FUNCTION
# ============================================================
def print_prediction(result, team1_name="Team 1", team2_name="Team 2"):
    """Pretty print prediction results"""
    print("\n" + "="*60)
    print(f"🎮 MATCH PREDICTION: {team1_name} vs {team2_name}")
    print("="*60)
    
    print(f"\n📊 Model Probabilities:")
    print(f"  {team1_name}: {result['team1_win_probability']:.1%}")
    print(f"  {team2_name}: {result['team2_win_probability']:.1%}")
    
    print(f"\n🏆 Predicted Winner: {result['predicted_winner']}")
    print(f"💪 Confidence: {result['confidence']:.1%}")
    
    if 'betting_analysis' in result:
        print(f"\n💰 BETTING ANALYSIS:")
        print("-" * 60)
        
        for team_key, team_name in [('team1', team1_name), ('team2', team2_name)]:
            bet_data = result['betting_analysis'][team_key]
            print(f"\n{team_name}:")
            print(f"  Odds: {bet_data['odds']:.2f}")
            print(f"  Model Probability: {bet_data['model_probability']:.1%}")
            print(f"  Implied Probability: {bet_data['implied_probability']:.1%}")
            print(f"  Edge: {bet_data['edge']:+.1%}")
            print(f"  Expected Value: {bet_data['expected_value']:+.1%}")
            print(f"  🎯 Recommendation: {bet_data['recommendation']}")
            
            if bet_data['recommendation'] == 'BET':
                print(f"     ✅ VALUE BET DETECTED!")


# ============================================================
# EXAMPLE USAGE
# ============================================================
if __name__ == "__main__":
    
    # Example 1: Strong favorite
    print("\n" + "#"*60)
    print("# EXAMPLE 1: T1 (strong) vs Weaker Team")
    print("#"*60)
    
    result1 = predict_match(
        # CORE ELO FEATURES
        team1_elo_before=1650,
        team2_elo_before=1480,
        
        # HEAD-TO-HEAD
        h2h_winrate=0.7,
        h2h_games_played=5,
        h2h_avg_score_diff=3.2,
        
        # RECENT FORM
        team1_winrate_3=0.83,
        team2_winrate_3=0.33,
        team1_winrate_10=0.75,
        team2_winrate_10=0.45,
        team1_winrate_30d=0.75,
        team2_winrate_30d=0.45,
        
        # ELO CONTEXT & MOMENTUM
        team1_elo_momentum=15,
        team2_elo_momentum=-8,
        team1_opponent_elo_mean=1580,
        team2_opponent_elo_mean=1520,
        
        # BETTING MARKET SIGNALS
        market_margin=0.04,
        
        # MATCH CONTEXT
        bo_type=5,
        is_playoffs=1,
        is_international=1,
        team1_tournament_games=25,
        team2_tournament_games=18,
        
        # SIDE PREFERENCE
        team1_blue_winrate=0.65,
        team1_red_winrate=0.60,
        team2_blue_winrate=0.48,
        team2_red_winrate=0.45,
        
        # REST & FATIGUE
        team1_days_rest=3,
        team2_days_rest=2,
        team1_games_30d=20,
        team2_games_30d=22,
        
        # IN-GAME PERFORMANCE
        team1_vision_control=1.2,
        team2_vision_control=0.9,
        team1_dragon_control=0.65,
        team2_dragon_control=0.45,
        team1_baron_control=0.70,
        team2_baron_control=0.40,
        
        # Odds
        team1_odds=1.35,
        team2_odds=3.20
    )
    
    print_prediction(result1, "T1", "Weaker Team")
    
    
    # Example 2: Even matchup
    print("\n\n" + "#"*60)
    print("# EXAMPLE 2: G2 vs FNC (even matchup)")
    print("#"*60)
    
    result2 = predict_match(
        # CORE ELO FEATURES
        team1_elo_before=1570,
        team2_elo_before=1560,
        
        # HEAD-TO-HEAD
        h2h_winrate=0.52,
        h2h_games_played=10,
        h2h_avg_score_diff=0.8,
        
        # RECENT FORM
        team1_winrate_3=0.67,
        team2_winrate_3=0.67,
        team1_winrate_10=0.60,
        team2_winrate_10=0.58,
        team1_winrate_30d=0.60,
        team2_winrate_30d=0.58,
        
        # ELO CONTEXT & MOMENTUM
        team1_elo_momentum=5,
        team2_elo_momentum=3,
        team1_opponent_elo_mean=1540,
        team2_opponent_elo_mean=1535,
        
        # BETTING MARKET SIGNALS
        market_margin=0.06,
        
        # MATCH CONTEXT
        bo_type=3,
        is_playoffs=0,
        is_international=0,
        team1_tournament_games=22,
        team2_tournament_games=20,
        
        # SIDE PREFERENCE
        team1_blue_winrate=0.55,
        team1_red_winrate=0.52,
        team2_blue_winrate=0.53,
        team2_red_winrate=0.50,
        
        # REST & FATIGUE
        team1_days_rest=4,
        team2_days_rest=4,
        team1_games_30d=18,
        team2_games_30d=19,
        
        # IN-GAME PERFORMANCE
        team1_vision_control=1.1,
        team2_vision_control=1.05,
        team1_dragon_control=0.55,
        team2_dragon_control=0.53,
        team1_baron_control=0.58,
        team2_baron_control=0.56,
        
        # Odds
        team1_odds=1.95,
        team2_odds=1.90
    )
    
    print_prediction(result2, "G2 Esports", "Fnatic")